
# Train on Jan–Nov 2023, Predict Dec 2023 (Power LSTM)

This notebook:
1. Trains an LSTM model on **January–November 2023** data.
2. Validates with the tail of that training window (chronological split).
3. Generates predictions for **December 2023** and computes metrics.
4. Saves artifacts (model + scalers + config) and CSV outputs.


In [24]:

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import r2_score
import joblib

print("TensorFlow:", tf.__version__)
print("Pandas:", pd.__version__)


TensorFlow: 2.20.0-rc0
Pandas: 2.2.3


In [ ]:

# ======================
# Configuration
# ======================

# Site / data location
#site = 'S0186'
#postcode = 5290

#site = 'S0561'
#postcode = 5555

site = 'S0556'
postcode = 5098

data_directory = '../Data/Processed'
site_filepath = os.path.join(data_directory, f"SA_site_edp_2023_{site}_processed.csv")
nci_path = "../Data/NCI/NCI_processed_grouped_all_SA_2023.csv"

# Time windows (local Australia/Adelaide)
tz = "Australia/Adelaide"
TRAIN_START = pd.Timestamp('2023-01-01 00:00:00', tz=tz)
TRAIN_END   = pd.Timestamp('2023-11-30 23:59:59', tz=tz)
PRED_START  = pd.Timestamp('2023-12-01 00:00:00', tz=tz)
PRED_END    = pd.Timestamp('2023-12-31 23:59:59', tz=tz)

# Modeling params
timesteps = 6                  # lookback length
merge_tolerance = "10min"
val_frac = 0.1                 # validation fraction from the *end* of the training window
use_past_power = True          # include 'Power' in inputs (autoregressive one-step-ahead)
mask_daylight_in_train = True  # train only on daylight sequences
use_solar_elevation_for_day = True
ghi_day_threshold = 20.0       # used if use_solar_elevation_for_day=False

# Artifacts/output
artifacts_dir = "./artifacts_power_lstm_2023_11train"
os.makedirs(artifacts_dir, exist_ok=True)
model_path = os.path.join(artifacts_dir, "best_lstm_power.keras")   # Keras 3 format
scaler_feat_path = os.path.join(artifacts_dir, "scaler_features.gz")
scaler_tgt_path  = os.path.join(artifacts_dir, "scaler_target.gz")  # only used when use_past_power=False
config_path = os.path.join(artifacts_dir, "config.json")

pred_csv_path = os.path.join(artifacts_dir, "power_predictions_2023_12.csv")
daily_metrics_csv_path = os.path.join(artifacts_dir, "power_daily_metrics_2023_12.csv")


In [26]:

# ======================
# Helpers
# ======================
def localize_adelaide_naive(series: pd.Series) -> pd.Series:
    """Localize naive timestamps to Australia/Adelaide with DST handling."""
    return series.dt.tz_localize(
        "Australia/Adelaide",
        ambiguous="NaT",
        nonexistent="shift_forward"
    )

def align_site_nci(site_filepath: str,
                   nci_path: str,
                   postcode: int,
                   start_time: pd.Timestamp,
                   end_time: pd.Timestamp,
                   merge_tolerance: str = "10min") -> pd.DataFrame:
    """Load site + NCI, align on time in Australia/Adelaide, return merged frame."""
    # Site
    df_site = pd.read_csv(site_filepath)
    if 'Power' not in df_site.columns:
        raise KeyError("Expected 'Power' column in site CSV.")
    df_site['datetime'] = pd.to_datetime(df_site['datetime'], errors='coerce')
    if df_site['datetime'].dt.tz is None:
        df_site['datetime'] = localize_adelaide_naive(df_site['datetime'])
    else:
        df_site['datetime'] = df_site['datetime'].dt.tz_convert('Australia/Adelaide')
    df_site = df_site[(df_site['datetime'] >= start_time) & (df_site['datetime'] <= end_time)].copy()
    if df_site.empty:
        raise ValueError("No site data in the selected time window after timezone alignment.")
    df_site = df_site.sort_values('datetime')

    # NCI
    nci = pd.read_csv(nci_path)
    nci = nci[nci['postcode'].notna()].copy()
    nci['postcode'] = nci['postcode'].astype(int)
    nci = nci[nci['postcode'] == postcode].copy()
    if nci.empty:
        raise ValueError(f"No NCI rows for postcode {postcode} in {nci_path}")
    nci['time'] = pd.to_datetime(nci['time'], errors='coerce', utc=True).dt.tz_convert('Australia/Adelaide')

    # Derive GHI if needed
    if 'surface_global_irradiance' not in nci.columns:
        req = {'direct_normal_irradiance','surface_diffuse_irradiance','solar_elevation'}
        if not req.issubset(nci.columns):
            raise KeyError("NCI missing 'surface_global_irradiance' and components to derive it.")
        nci['zenith_angle'] = 90 - nci['solar_elevation']
        nci['surface_global_irradiance'] = (
            nci['surface_diffuse_irradiance']
            + nci['direct_normal_irradiance'] * np.cos(np.radians(nci['zenith_angle']))
        )

    keep = [
        'time',
        'surface_global_irradiance',
        'direct_normal_irradiance',
        'surface_diffuse_irradiance',
        'cloud_type',
        'cloud_optical_depth',
        'solar_elevation',
        'solar_azimuth',
    ]
    for c in keep:
        if c not in nci.columns:
            raise KeyError(f"NCI missing required column: {c}")

    nci_small = nci[keep].sort_values('time')

    # Align (nearest within tolerance)
    merged = pd.merge_asof(
        df_site, nci_small,
        left_on='datetime', right_on='time',
        direction='nearest', tolerance=pd.Timedelta(merge_tolerance)
    ).drop(columns=['time'])

    # Interpolate NCI cols
    for c in ['surface_global_irradiance','direct_normal_irradiance','surface_diffuse_irradiance',
              'cloud_type','cloud_optical_depth','solar_elevation','solar_azimuth']:
        merged[c] = merged[c].interpolate(limit_direction='both')

    return merged

def create_sequences(data2d: np.ndarray, steps: int, target_col: int):
    """Return X, y, row_idx for one-step-ahead sequences."""
    X, y, idx = [], [], []
    for i in range(steps, len(data2d)):
        X.append(data2d[i-steps:i, :])
        y.append(data2d[i, target_col])
        idx.append(i)
    return np.array(X), np.array(y).reshape(-1,1), np.array(idx)

def make_sequences_for_inference(features_scaled: np.ndarray, timesteps: int):
    X, row_idx = [], []
    for i in range(timesteps, len(features_scaled)):
        X.append(features_scaled[i-timesteps:i, :])
        row_idx.append(i)
    return np.array(X), np.array(row_idx)

def inverse_minmax_column(pred_scaled: np.ndarray, scaler: MinMaxScaler, col_idx: int) -> np.ndarray:
    data_min = scaler.data_min_[col_idx]
    data_max = scaler.data_max_[col_idx]
    return pred_scaled * (data_max - data_min) + data_min


In [27]:

# ======================
# Load & prepare TRAIN (Jan–Nov 2023)
# ======================
merged_train = align_site_nci(
    site_filepath=site_filepath,
    nci_path=nci_path,
    postcode=postcode,
    start_time=TRAIN_START,
    end_time=TRAIN_END,
    merge_tolerance=merge_tolerance
)

# Feature columns
if use_past_power:
    feature_cols = [
        'direct_normal_irradiance',
        'surface_diffuse_irradiance',
        'cloud_type',
        'cloud_optical_depth',
        'solar_elevation',
        'solar_azimuth',
        'surface_global_irradiance',
        'Power',  # autoregressive input
    ]
else:
    feature_cols = [
        'direct_normal_irradiance',
        'surface_diffuse_irradiance',
        'cloud_type',
        'cloud_optical_depth',
        'solar_elevation',
        'solar_azimuth',
        'surface_global_irradiance',
    ]

target_name = 'Power'
for c in feature_cols + [target_name]:
    if c not in merged_train.columns:
        raise KeyError(f"Missing column in training data: {c}")

# Raw arrays
features_raw_train = merged_train[feature_cols].ffill().bfill().to_numpy(dtype=np.float64)
target_raw_train   = merged_train[[target_name]].ffill().bfill().to_numpy(dtype=np.float64)

# Chronological split inside Jan–Nov block
N_tr = len(features_raw_train)
split_row = int(N_tr * (1 - val_frac))
if split_row <= timesteps:
    raise ValueError("Not enough rows in training for chosen timesteps/val split.")

# Scale features (fit on train-only slice)
scaler_features = MinMaxScaler()
scaler_features.fit(features_raw_train[:split_row])
features_scaled_train = scaler_features.transform(features_raw_train)

# Target scaling (if target not included in features)
if use_past_power:
    target_in_features = True
    target_col_idx = feature_cols.index(target_name)
    scaler_target = None
else:
    target_in_features = False
    target_col_idx = None
    scaler_target = MinMaxScaler()
    scaler_target.fit(target_raw_train[:split_row])
    target_scaled_train = scaler_target.transform(target_raw_train)

# Sequences across whole Jan–Nov block
if use_past_power:
    X_all_tr, y_all_tr, row_idx_tr = create_sequences(features_scaled_train, timesteps, target_col_idx)
else:
    # dummy to get row_idx, y from target_scaled
    X_all_tr, _, row_idx_tr = create_sequences(features_scaled_train, timesteps, target_col=0)
    y_all_tr = np.array([target_scaled_train[i, 0] for i in row_idx_tr]).reshape(-1, 1)

# Daylight mask for TRAIN only
if mask_daylight_in_train:
    if use_solar_elevation_for_day:
        daylight_row = (merged_train['solar_elevation'].to_numpy() > 0)
    else:
        daylight_row = (merged_train['surface_global_irradiance'].to_numpy() > ghi_day_threshold)

    day_series = pd.Series(daylight_row.astype(int))
    all_inputs_day = (
        day_series.shift(1)  # window ends at i-1
                 .rolling(window=timesteps, min_periods=timesteps)
                 .min()
                 .fillna(0)
                 .astype(bool)
                 .to_numpy()
    )
    target_is_day = daylight_row
    sample_ok = target_is_day & all_inputs_day
    seq_ok_tr = sample_ok[row_idx_tr]
else:
    seq_ok_tr = np.ones_like(row_idx_tr, dtype=bool)

# Split into train/val (chronological)
train_mask_time = (row_idx_tr < split_row)
val_mask_time   = (row_idx_tr >= split_row)
train_mask = train_mask_time & seq_ok_tr
val_mask   = val_mask_time   # keep evening/night in validation to measure robustness

X_train, y_train = X_all_tr[train_mask], y_all_tr[train_mask]
X_val,   y_val   = X_all_tr[val_mask],   y_all_tr[val_mask]

X_train = X_train.astype(np.float32); y_train = y_train.astype(np.float32)
X_val   = X_val.astype(np.float32);   y_val   = y_val.astype(np.float32)

print(f"Jan–Nov rows: {N_tr:,}")
print(f"Train samples: {X_train.shape[0]}  (kept {train_mask.sum()} / {train_mask_time.sum()} after daylight mask)")
print(f"Val   samples: {X_val.shape[0]}    (kept {val_mask.sum()} / {val_mask_time.sum()})")


Jan–Nov rows: 96,126
Train samples: 53804  (kept 53804 / 86507 after daylight mask)
Val   samples: 9613    (kept 9613 / 9613)


In [28]:

# ======================
# Build datasets & train model
# ======================
batch_size = 512
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_ds   = tf.data.Dataset.from_tensor_slices((X_val,   y_val)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

n_features = X_train.shape[-1]
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(timesteps, n_features)),
    Dropout(0.2),
    LSTM(32),
    Dense(1),
])
model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss=tf.keras.losses.MeanSquaredError())

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint(model_path, save_best_only=True)  # saves .keras

history = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=[early_stop, checkpoint])

# Save scalers + config
joblib.dump(scaler_features, scaler_feat_path)
if not target_in_features:
    joblib.dump(scaler_target, scaler_tgt_path)

config = {
    "timesteps": timesteps,
    "feature_cols": feature_cols,
    "target_name": "Power",
    "target_in_features": target_in_features,
    "target_col_idx": target_col_idx,
    "merge_tolerance": merge_tolerance,
    "mask_daylight_in_train": mask_daylight_in_train,
    "use_solar_elevation_for_day": use_solar_elevation_for_day,
    "ghi_day_threshold": ghi_day_threshold,
    "train_window": [str(TRAIN_START), str(TRAIN_END)],
    "predict_window": [str(PRED_START), str(PRED_END)],
}
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("Saved artifacts to:", artifacts_dir)


Epoch 1/20


c:\Users\z3548018\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


106/106 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 0.0227 - val_loss: 0.0047
Epoch 2/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 0.0078 - val_loss: 0.0043
Epoch 3/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0074 - val_loss: 0.0042
Epoch 4/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0072 - val_loss: 0.0040
Epoch 5/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0070 - val_loss: 0.0039
Epoch 6/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0067 - val_loss: 0.0037
Epoch 7/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0066 - val_loss: 0.0036
Epoch 8/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0064 - val_loss: 0.0035
Epoch 9/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0063 - val_loss: 0.0033
Epoch 10/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0061 - val_loss: 0.0033
Epoch 11/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0058 - val_loss: 0.0032
Epoch 12/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/ste

In [29]:

# ======================
# Predict December 2023
# ======================
# Reload model (best checkpoint)
model = tf.keras.models.load_model(model_path)

merged_pred = align_site_nci(
    site_filepath=site_filepath,
    nci_path=nci_path,
    postcode=postcode,
    start_time=PRED_START,
    end_time=PRED_END,
    merge_tolerance=merge_tolerance
)

# Check columns
for c in feature_cols + ["Power"]:
    if c not in merged_pred.columns:
        raise KeyError(f"Missing column in prediction data: {c}")

# Scale using training scaler
features_raw_pred = merged_pred[feature_cols].ffill().bfill().to_numpy(dtype=np.float64)
features_scaled_pred = scaler_features.transform(features_raw_pred)

# Build sequences for inference (all rows, incl. night)
def make_sequences_for_inference(features_scaled: np.ndarray, timesteps: int):
    X, row_idx = [], []
    for i in range(timesteps, len(features_scaled)):
        X.append(features_scaled[i-timesteps:i, :])
        row_idx.append(i)
    return np.array(X), np.array(row_idx)

X_inf, row_idx = make_sequences_for_inference(features_scaled_pred, timesteps)
X_inf = X_inf.astype(np.float32)

# Predict (scaled)
batch_size_inf = 2048
inf_ds = tf.data.Dataset.from_tensor_slices(X_inf).batch(batch_size_inf).prefetch(tf.data.AUTOTUNE)
pred_scaled = model.predict(inf_ds, verbose=0).reshape(-1)

# Inverse-scale to file units
def inverse_minmax_column(pred_scaled: np.ndarray, scaler: MinMaxScaler, col_idx: int) -> np.ndarray:
    data_min = scaler.data_min_[col_idx]
    data_max = scaler.data_max_[col_idx]
    return pred_scaled * (data_max - data_min) + data_min

if target_in_features:
    power_pred = inverse_minmax_column(pred_scaled, scaler_features, col_idx=target_col_idx)
else:
    if os.path.exists(scaler_tgt_path):
        scaler_target = joblib.load(scaler_tgt_path)
        power_pred = scaler_target.inverse_transform(pred_scaled.reshape(-1,1)).reshape(-1)
    else:
        raise RuntimeError("Target scaler not found for non-autoregressive model.")

# Assemble results
times_all = merged_pred['datetime'].reset_index(drop=True)
times_pred = times_all.iloc[row_idx].reset_index(drop=True)
results = pd.DataFrame({
    "datetime": times_pred,
    "Power_pred_W": power_pred,
    "Power_true_W": merged_pred['Power'].to_numpy()[row_idx]
})

# Metrics (all times + daylight-only)
def _metrics(y_t, y_p):
    mae  = float(np.mean(np.abs(y_p - y_t)))
    rmse = float(np.sqrt(np.mean((y_p - y_t) ** 2)))
    r2   = float(r2_score(y_t, y_p)) if len(y_t) > 1 else float("nan")
    return mae, rmse, r2

y_true = results["Power_true_W"].to_numpy()
y_pred = results["Power_pred_W"].to_numpy()
mae_all, rmse_all, r2_all = _metrics(y_true, y_pred)

solar_elev_seq = merged_pred['solar_elevation'].to_numpy()[row_idx]
day_mask = solar_elev_seq > 0
if day_mask.any():
    mae_day, rmse_day, r2_day = _metrics(y_true[day_mask], y_pred[day_mask])
else:
    mae_day = rmse_day = r2_day = float("nan")

print("December 2023 metrics")
print(f"ALL times -> MAE: {mae_all:,.6f}  RMSE: {rmse_all:,.6f}  R²: {r2_all:,.4f}")
print(f"DAYLIGHT  -> MAE: {mae_day:,.6f}  RMSE: {rmse_day:,.6f}  R²: {r2_day:,.4f}")

# Save outputs
results.to_csv(pred_csv_path, index=False)

# Daily metrics
results['date'] = results['datetime'].dt.date
results['daylight'] = day_mask
def _agg_all(g):
    m, r, r2 = _metrics(g['Power_true_W'].to_numpy(), g['Power_pred_W'].to_numpy())
    return pd.Series({'MAE': m, 'RMSE': r, 'R2': r2, 'N': len(g)})
def _agg_day(g):
    mask = g['daylight'].to_numpy()
    if mask.any():
        m, r, r2 = _metrics(g.loc[mask,'Power_true_W'].to_numpy(), g.loc[mask,'Power_pred_W'].to_numpy())
        n = int(mask.sum())
    else:
        m = r = r2 = float("nan"); n = 0
    return pd.Series({'MAE_day': m, 'RMSE_day': r, 'R2_day': r2, 'N_day': n})

daily_all = results.groupby('date', sort=True).apply(_agg_all).reset_index()
daily_day = results.groupby('date', sort=True).apply(_agg_day).reset_index()
daily_metrics = pd.merge(daily_all, daily_day, on='date', how='left')
daily_metrics.to_csv(daily_metrics_csv_path, index=False)

print("\nSaved:")
print("  Predictions ->", pred_csv_path)
print("  Daily metrics ->", daily_metrics_csv_path)


December 2023 metrics
ALL times -> MAE: 0.167518  RMSE: 0.302324  R²: 0.9618
DAYLIGHT  -> MAE: 0.205048  RMSE: 0.345712  R²: 0.9543

Saved:
  Predictions -> ./artifacts_power_lstm_2023_11train\power_predictions_2023_12.csv
  Daily metrics -> ./artifacts_power_lstm_2023_11train\power_daily_metrics_2023_12.csv


C:\Users\z3548018\AppData\Local\Temp\ipykernel_14780\406749180.py:105: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_all = results.groupby('date', sort=True).apply(_agg_all).reset_index()
C:\Users\z3548018\AppData\Local\Temp\ipykernel_14780\406749180.py:106: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_day = results.groupby('date', sort=True).apply(_agg_day).reset_index()


In [30]:
# =========================================
# Plot ALL December 2023 daily Predicted vs Actual power
# + polyfit line (06:00–20:00 local)  + GHI overlay (right y-axis)
# Clamp all plotted series at 0 (no negatives shown)
# =========================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# ---- Paths (use the one from earlier cell if available) ----
try:
    csv_path = pred_csv_path
except NameError:
    # Fallback if running standalone; adjust if your artifacts_dir is different
    csv_path = "./artifacts_power_lstm_2023_11train/power_predictions_2023_12.csv"

# Input data paths for pulling GHI via align_site_nci()
# (these should match what you used during training/prediction)
try:
    data_directory
except NameError:
    data_directory = '../Data/Processed'
try:
    nci_path
except NameError:
    nci_path = "../Data/NCI/NCI_processed_grouped_all_SA_2023.csv"

site_filepath = os.path.join(data_directory, f"SA_site_edp_2023_{site}_processed.csv")

# allow site to be interpolated in folder name if it's already defined
try:
    output_dir = f"./december_daily_plots_{site}_Polyfit"
except NameError:
    output_dir = "./december_daily_plots_Polyfit"

pdf_path = os.path.join(output_dir, "power_december_2023_plots.pdf")
show_plots = False  # set True to display as they are created

# Polyfit options
poly_degree = 2
tz = "Australia/Adelaide"
poly_start_h = 6    # 06:00 (inclusive)
poly_end_h   = 20   # 20:00 (inclusive)

# ---- Helper: robust timezone handling ----
def to_adelaide_series_any(df: pd.DataFrame, col: str) -> pd.Series:
    s = pd.to_datetime(df[col], errors="coerce", utc=True)
    if s.notna().any():
        return s.dt.tz_convert(tz)
    s = pd.to_datetime(df[col], errors="coerce")
    if s.isna().all():
        raise ValueError(f"Could not parse any datetimes from '{col}'.")
    tzinfo = getattr(s.dtype, "tz", None)
    if tzinfo is None:
        return s.dt.tz_localize(tz)
    return s.dt.tz_convert(tz)

# ---- Load predictions ----
df = pd.read_csv(csv_path)
if "datetime" not in df.columns:
    raise KeyError("Expected 'datetime' column in predictions CSV.")
if "Power_pred_W" not in df.columns:
    raise KeyError("Expected 'Power_pred_W' column in predictions CSV.")

df["datetime"] = to_adelaide_series_any(df, "datetime")

# ---- Filter to Dec 2023 (inclusive) ----
start_ts = pd.Timestamp("2023-12-01 00:00:00", tz=tz)
end_ts   = pd.Timestamp("2023-12-31 23:59:59", tz=tz)
mask = (df["datetime"] >= start_ts) & (df["datetime"] <= end_ts)
dfr = df.loc[mask].copy().sort_values("datetime")
if dfr.empty:
    raise ValueError("No rows found for December 2023 in the predictions file.")

# ---- Pull GHI for the same window using your existing align_site_nci() helper ----
# Requires align_site_nci() to be defined in the notebook/session
dec_merged_for_ghi = align_site_nci(
    site_filepath=site_filepath,
    nci_path=nci_path,
    postcode=postcode,
    start_time=start_ts,
    end_time=end_ts,
    merge_tolerance="10min"
)
ghi_df = dec_merged_for_ghi[["datetime", "surface_global_irradiance"]].copy()
ghi_df = ghi_df.sort_values("datetime").reset_index(drop=True)

# ---- Plot per day ----
os.makedirs(output_dir, exist_ok=True)
has_actual = "Power_true_W" in dfr.columns

# add a local calendar date column for grouping
dfr["date"] = dfr["datetime"].dt.date
ghi_df["date"] = ghi_df["datetime"].dt.date

with PdfPages(pdf_path) as pdf:
    for date_value, g in dfr.groupby("date", sort=True):
        g = g.sort_values("datetime")

        # choose which series to fit (prefer actual if we have it)
        y_fit_col = "Power_true_W" if has_actual else "Power_pred_W"
        y_fit_label_suffix = "Actual" if has_actual else "Pred"

        # --- define the polyfit window: poly_start_h to poly_end_h local ---
        day_start = pd.Timestamp(f"{date_value} {poly_start_h:02d}:00:00", tz=tz)
        day_end   = pd.Timestamp(f"{date_value} {poly_end_h:02d}:00:00",   tz=tz)

        # keep data only inside the fitting window for polyfit
        gp = g[(g["datetime"] >= day_start) & (g["datetime"] <= day_end)].copy()

        # prepare x as seconds since poly_start_h (helps conditioning)
        t0 = day_start
        if not gp.empty:
            x_fit = (gp["datetime"] - t0).dt.total_seconds().to_numpy()
            y_fit = gp[y_fit_col].to_numpy()
        else:
            x_fit = np.array([])
            y_fit = np.array([])

        # drop NaNs for fitting
        valid = np.isfinite(x_fit) & np.isfinite(y_fit)
        poly_added = False
        y_poly_eval = None

        # Fit only if we have enough valid points
        if valid.sum() >= (poly_degree + 1):
            try:
                coeffs = np.polyfit(x_fit[valid], y_fit[valid], deg=poly_degree)
                # Evaluate polynomial across the same [start, end] span
                seconds_span = int((day_end - day_start).total_seconds())
                x_eval = np.linspace(0, seconds_span, 721)  # ~one-minute resolution
                y_poly_eval = np.poly1d(coeffs)(x_eval)
                # ---- CLAMP polyfit values at 0 for plotting ----
                y_poly_eval = np.clip(y_poly_eval, 0, None)
                dt_eval = day_start + pd.to_timedelta(x_eval, unit="s")
                poly_added = True
            except Exception:
                poly_added = False

        # --- slice GHI for the same day (full day) ---
        ghi_day = ghi_df.loc[ghi_df["date"] == date_value, ["datetime", "surface_global_irradiance"]]
        has_ghi = not ghi_day.empty and ghi_day["surface_global_irradiance"].notna().any()
        if has_ghi:
            # ---- CLAMP GHI at 0 for plotting ----
            ghi_vals = np.clip(ghi_day["surface_global_irradiance"].to_numpy(), 0, None)

        # ---- CLAMP predicted/actual series at 0 for plotting ----
        pred_clamped = np.clip(g["Power_pred_W"].to_numpy(), 0, None)
        if has_actual:
            true_clamped = np.clip(g["Power_true_W"].to_numpy(), 0, None)

        # --- plotting with twin y-axis for GHI ---
        fig, ax1 = plt.subplots()

        # predicted line (entire day, clamped)
        l_pred, = ax1.plot(g["datetime"], pred_clamped, label="Predicted Power")
        lines = [l_pred]
        labels = ["Predicted Power"]

        # actual line if present (clamped)
        if has_actual:
            l_act, = ax1.plot(g["datetime"], true_clamped, label="Actual Power")
            lines.append(l_act)
            labels.append("Actual Power")

        # polyfit line (only between start and end, already clamped)
        if poly_added and y_poly_eval is not None:
            l_poly, = ax1.plot(dt_eval, y_poly_eval, linestyle="--",
                               label=f"Polyfit")
            lines.append(l_poly)
            labels.append(f"Polyfit")

        # GHI on right axis (clamped)
        if has_ghi:
            ax2 = ax1.twinx()
            l_ghi, = ax2.plot(ghi_day["datetime"], ghi_vals, linestyle=":", label="GHI")
            ax2.set_ylabel("GHI (W/m²)")
            lines.append(l_ghi)
            labels.append("GHI")

        ax1.set_title(f"Power — {date_value} (Australia/Adelaide)")
        ax1.set_xlabel("Time")
        ax1.set_ylabel("Power (file units)")
        ax1.grid(True)

        # unified legend from both axes
        ax1.legend(lines, labels, loc="best")

        fig.tight_layout()

        # Save PNG
        png_name = os.path.join(output_dir, f"power_{date_value}.png")
        fig.savefig(png_name, dpi=150)
        # Save to combined PDF
        pdf.savefig(fig)
        if show_plots:
            plt.show()
        plt.close(fig)

print(f"Saved daily PNGs to: {os.path.abspath(output_dir)}")
print(f"Saved combined PDF to: {os.path.abspath(pdf_path)}")


Saved daily PNGs to: c:\Git\CICADAResearch\Curtailment\december_daily_plots_S0561_Polyfit
Saved combined PDF to: c:\Git\CICADAResearch\Curtailment\december_daily_plots_S0561_Polyfit\power_december_2023_plots.pdf
